# Gloved vs Ungloved Hand Detection
## Complete Training and Inference Pipeline

This notebook demonstrates:
1. Dataset preparation
2. Model training with YOLOv8
3. Inference and evaluation
4. Output generation (annotated images + JSON logs)

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install ultralytics opencv-python pillow matplotlib tqdm

import os
import json
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Setup complete!")

## 2. Dataset Preparation

### Recommended Datasets:
- **Roboflow Universe**: Search for "glove detection" or "PPE hands"
- **Custom Dataset**: Hand images with glove/no-glove annotations

Dataset should be in YOLO format:
```
dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

In [ ]:
# Dataset configuration
dataset_config = """
# Dataset configuration for YOLO
path: ./dataset  # dataset root dir
train: images/train  # train images
val: images/val  # val images

# Classes
nc: 2  # number of classes
names: ['gloved_hand', 'bare_hand']  # class names
"""

# Save config
with open('dataset.yaml', 'w') as f:
    f.write(dataset_config)

print("Dataset config created!")

## 3. Model Training

Train YOLOv8 on glove detection dataset

In [ ]:
# Initialize YOLOv8 model
model = YOLO('yolov8n.pt')  # Use nano for speed, 'm' or 'l' for better accuracy

# Train the model
results = model.train(
    data='dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='glove_detector',
    patience=10,
    save=True,
    # Data augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0
)

print("Training complete!")

## 4. Model Evaluation

In [ ]:
# Validate the model
metrics = model.val()

print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

## 5. Inference Pipeline

In [ ]:
class GloveDetector:
    """Glove detection inference class"""
    
    def __init__(self, model_path, confidence=0.5):
        self.model = YOLO(model_path)
        self.confidence = confidence
        self.colors = {
            'gloved_hand': (0, 255, 0),
            'bare_hand': (0, 0, 255)
        }
    
    def detect(self, image_path):
        """Detect gloves in image"""
        image = cv2.imread(image_path)
        results = self.model(image, conf=self.confidence, verbose=False)
        
        detections = []
        
        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = float(box.conf[0])
                cls = int(box.cls[0])
                label = result.names[cls]
                
                detection = {
                    'label': label,
                    'confidence': round(conf, 2),
                    'bbox': [int(x1), int(y1), int(x2), int(y2)]
                }
                detections.append(detection)
                
                # Draw on image
                color = self.colors.get(label, (255, 255, 255))
                cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                cv2.putText(image, f"{label}: {conf:.2f}", (int(x1), int(y1)-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        return image, detections

# Initialize detector with trained model
detector = GloveDetector('runs/detect/glove_detector/weights/best.pt', confidence=0.5)
print("Detector initialized!")

## 6. Process Images and Generate Outputs

In [ ]:
def process_folder(input_folder, output_folder, logs_folder):
    """Process all images and save results"""
    
    # Create directories
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    Path(logs_folder).mkdir(parents=True, exist_ok=True)
    
    # Get image files
    image_files = list(Path(input_folder).glob('*.jpg')) + \
                  list(Path(input_folder).glob('*.png'))
    
    print(f"Processing {len(image_files)} images...")
    
    for img_path in tqdm(image_files):
        # Detect
        annotated_img, detections = detector.detect(str(img_path))
        
        # Save annotated image
        output_path = Path(output_folder) / img_path.name
        cv2.imwrite(str(output_path), annotated_img)
        
        # Save JSON log
        log_data = {
            'filename': img_path.name,
            'detections': detections
        }
        log_path = Path(logs_folder) / f"{img_path.stem}_detections.json"
        with open(log_path, 'w') as f:
            json.dump(log_data, f, indent=2)
    
    print("Processing complete!")

# Run processing
process_folder('./test_images', './output', './logs')

## 7. Visualize Results

In [ ]:
# Display sample results
output_images = list(Path('./output').glob('*.jpg'))[:5]

fig, axes = plt.subplots(1, len(output_images), figsize=(20, 4))

for ax, img_path in zip(axes, output_images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(img_path.name)

plt.tight_layout()
plt.show()

## 8. Performance Analysis

In [ ]:
# Analyze detection statistics
log_files = list(Path('./logs').glob('*.json'))

total_detections = 0
gloved_count = 0
bare_count = 0

for log_file in log_files:
    with open(log_file, 'r') as f:
        data = json.load(f)
        for det in data['detections']:
            total_detections += 1
            if det['label'] == 'gloved_hand':
                gloved_count += 1
            elif det['label'] == 'bare_hand':
                bare_count += 1

print(f"Total detections: {total_detections}")
print(f"Gloved hands: {gloved_count}")
print(f"Bare hands: {bare_count}")